2. Binarization: Binarization converts numerical values into only two values: 0 and 1, based on a threshold.

Why do we use it?
To convert data into binary form.
Useful for algorithms that work with binary features.
Simplifies decision-making.

Where is it used?
Email spam detection (Spam = 1, Not Spam = 0)
Pass/Fail classification
Disease prediction (Disease = 1, No Disease = 0)
Image processing (Black = 0, White = 1)

In [1]:
import numpy as np
import pandas as pd

In [2]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [3]:
import kagglehub

path = kagglehub.dataset_download("hesh97/titanicdataset-traincsv")

print(path)

C:\Users\Dir com\.cache\kagglehub\datasets\hesh97\titanicdataset-traincsv\versions\1


In [4]:
import os
# Import the required libraries before using them
df = pd.read_csv(os.path.join(path, os.listdir(path)[0]))
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df = df[['Age', 'Fare', 'SibSp'	, 'Parch', 'Survived']]

df.head()

,Age,Fare,SibSp,Parch,Survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [6]:
#joining Two Columns
df['family'] = df['SibSp'] + df['Parch']

In [7]:
df.head()

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [8]:
#Droping Columns
df.drop(columns= ['SibSp' ,'Parch'],inplace=True)

In [9]:
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [10]:
x = df.drop(columns=['Survived'])   
y = df['Survived']

In [11]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

In [12]:
x_train.head()

,Age,Fare,family
331,45.5,28.5000,0
733,23.0,13.0000,0
382,32.0,7.9250,0
704,26.0,7.8542,1
813,6.0,31.2750,6


In [13]:
# Without Binarization
clf = DecisionTreeClassifier()
clf.fit(x_train, y_train)
y_pred = clf.predict(x_test)
accuracy_score(y_test , y_pred)

0.659217877094972

In [14]:
np.mean(
    cross_val_score(
        DecisionTreeClassifier(),
        x,y,cv=10,scoring='accuracy'
)
)

np.float64(0.6509612983770288)

In [15]:
# Applying Binarization
from sklearn.preprocessing import Binarizer

In [17]:
trf = ColumnTransformer([
    ('bin' , Binarizer(copy=False),['family'])
],remainder='passthrough')

In [18]:
x_train_trf = trf.fit_transform(x_train)
x_test_trf = trf.transform(x_test)

In [21]:
pd.DataFrame(x_train_trf,columns=['family' , 'Age' , 'Fare'])

,family,Age,Fare
0,0.0,45.5,28.5000
1,0.0,23.0,13.0000
2,0.0,32.0,7.9250
3,1.0,26.0,7.8542
4,1.0,6.0,31.2750
...,...,...,...
707,0.0,21.0,7.6500
708,0.0,NaN,31.0000
709,1.0,41.0,14.1083
710,1.0,14.0,120.0000


In [22]:
clf = DecisionTreeClassifier()
clf.fit(x_train, y_train)
y_pred2 = clf.predict(x_test)
accuracy_score(y_test , y_pred2)

0.6368715083798883

In [25]:
x_trf = trf.fit_transform(x)

score = np.mean(
    cross_val_score(
        DecisionTreeClassifier(random_state=42),
        x_trf,
        y,
        cv=10,
        scoring='accuracy'
    )
)

print(score)

0.6520973782771536
